## Netflix Dataset Analysis

### 1. Load Data and Initial Exploration

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load the dataset from the local file
print("Attempting to load Netflix dataset from 'netflix_titles.csv'...")
try:
    df = pd.read_csv('netflix_titles.csv')
    print("Netflix dataset loaded successfully from local file.")
except FileNotFoundError:
    print("Error: 'netflix_titles.csv' not found. Please ensure the file is present in the current directory.")
    df = pd.DataFrame() # Create an empty DataFrame if file not found
except Exception as e:
    print(f"An error occurred while loading the dataset: {e}")
    df = pd.DataFrame() # Create an empty DataFrame if an error occurs

if not df.empty:
    print("Dataset loaded successfully.")
    display(df.head())
    display(df.info())
    display(df.describe(include='all'))
else:
    print("No data loaded. Please ensure the dataset is available or provide a correct path/URL.")

Attempting to load Netflix dataset from 'netflix_titles.csv'...
Error: 'netflix_titles.csv' not found. Please ensure the file is present in the current directory.
No data loaded. Please ensure the dataset is available or provide a correct path/URL.


### 2. Data Cleaning and Preparation

Based on the `df.info()` output, we might need to handle missing values, convert data types, and potentially create new features for analysis.

In [2]:
if not df.empty:
    # Convert 'date_added' to datetime objects
    df['date_added'] = pd.to_datetime(df['date_added'], errors='coerce')

    # Extract year added for trend analysis
    df['year_added'] = df['date_added'].dt.year

    # Fill missing 'country' with 'Unknown' (or mode, depending on analysis goal)
    df['country'].fillna('Unknown', inplace=True)

    # Fill missing 'rating' with mode
    df['rating'].fillna(df['rating'].mode()[0], inplace=True)

    # Drop rows where 'date_added' could not be parsed (if any)
    df.dropna(subset=['date_added'], inplace=True)

    print("\nAfter cleaning and preparation:")
    display(df.info())
    display(df.head())
else:
    print("Cannot perform data cleaning as no data was loaded.")

Cannot perform data cleaning as no data was loaded.


### 3. Summary Statistics

Now we can look at summary statistics of key categorical and numerical features.

In [3]:
if not df.empty:
    print("\nSummary statistics for key columns:")
    display(df['type'].value_counts())
    display(df['rating'].value_counts())
    display(df['country'].value_counts().head(10))
    display(df['release_year'].describe())
    display(df['duration'].apply(lambda x: int(x.split(' ')[0]) if isinstance(x, str) and 'min' in x else None).dropna().describe()) # For movies duration
else:
    print("Cannot calculate summary statistics as no data was loaded.")

Cannot calculate summary statistics as no data was loaded.


### 4. Trend Identification (Content Added Over Time)

Let's analyze the trends in content added to Netflix over the years.

In [4]:
if not df.empty:
    content_added_by_year = df.groupby('year_added').size().reset_index(name='count')

    plt.figure(figsize=(12, 6))
    sns.lineplot(x='year_added', y='count', data=content_added_by_year)
    plt.title('Number of Titles Added to Netflix Over Time')
    plt.xlabel('Year Added')
    plt.ylabel('Number of Titles')
    plt.grid(True)
    plt.show()

    content_type_by_year = df.groupby(['year_added', 'type']).size().unstack(fill_value=0)

    content_type_by_year.plot(kind='line', figsize=(12, 6))
    plt.title('Movies vs TV Shows Added to Netflix Over Time')
    plt.xlabel('Year Added')
    plt.ylabel('Number of Titles')
    plt.grid(True)
    plt.legend(title='Content Type')
    plt.show()
else:
    print("Cannot identify trends as no data was loaded.")

Cannot identify trends as no data was loaded.


### 5. Correlation Analysis

For correlation analysis, we typically look at numerical features. In this dataset, `release_year` and `year_added` are the primary numerical candidates. We can also explore correlations between categorical features using methods like chi-squared tests or by visualizing their distributions.

In [5]:
if not df.empty:
    # Numerical correlation (if more numerical features existed)
    numerical_df = df[['release_year', 'year_added']]
    print("\nCorrelation matrix for numerical features:")
    display(numerical_df.corr())

    # Example: Relationship between 'type' and 'rating'
    plt.figure(figsize=(10, 6))
    sns.countplot(data=df, x='rating', hue='type', palette='viridis', order=df['rating'].value_counts().index)
    plt.title('Content Type Distribution by Rating')
    plt.xlabel('Rating')
    plt.ylabel('Count')
    plt.xticks(rotation=45)
    plt.show()

    # Example: Top 10 countries by content type
    top_countries = df['country'].value_counts().head(10).index
    df_top_countries = df[df['country'].isin(top_countries)]

    plt.figure(figsize=(14, 7))
    sns.countplot(data=df_top_countries, x='country', hue='type', palette='magma', order=top_countries)
    plt.title('Content Type Distribution in Top 10 Countries')
    plt.xlabel('Country')
    plt.ylabel('Count')
    plt.xticks(rotation=45)
    plt.show()
else:
    print("Cannot perform correlation analysis as no data was loaded.")

Cannot perform correlation analysis as no data was loaded.


### Next Steps / Further Exploration:

*   **Detailed Content Analysis**: Explore genres, directors, and actors to identify popular categories or collaborations.
*   **Duration Analysis**: Analyze movie durations and TV show season counts.
*   **Text Analysis**: For 'description' or 'listed_in' columns, natural language processing (NLP) techniques could reveal thematic trends.
*   **Audience Segmentation**: Combine rating with other features to infer audience preferences.